# DBKT - Bài Tập 2: Tải Dữ Liệu CPI & Chuyển Đổi Tần Suất Dữ Liệu trên R
## Thu Thập Và Chuẩn Hóa Dữ Liệu CPI Từ FRED, Ghép Chuỗi Với GDP

### 1. Tải dữ liệu CPI từ FRED
Nạp các thư viện `fredr` và `tidyverse`, cài đặt API key và tải dữ liệu Chỉ số giá tiêu dùng CPI (`CPIAUCSL`) từ năm 1970 đến 2025.

In [ ]:
library(fredr)
library(tidyverse)

# Cài đặt FRED API Key
fredr_set_key("b714d7951692a0472a8b387b99497510")

# Tải dữ liệu CPIAUCSL
cpi <- fredr(
  series_id = "CPIAUCSL",
  observation_start = as.Date("1970-01-01"),
  observation_end = as.Date("2025-01-01")
)

head(cpi)

### 9. Chuyển đổi tần suất dữ liệu
#### 9.1 Chuyển dữ liệu tháng sang quý
Sử dụng `zoo::as.yearqtr` để đổi mốc thời gian `date` sang quý và nhóm dữ liệu để lấy CPI trung bình theo từng quý.

In [ ]:
library(zoo)
library(dplyr)

# Chuyển đổi mốc thời gian sang dạng Quý (as.yearqtr)
cpi$quarter <- as.yearqtr(cpi$date)

# Tổng hợp theo quý (tính trung bình CPI)
cpi_q <- cpi |> 
  group_by(quarter) |> 
  summarise(cpi = mean(value, na.rm = TRUE))

head(cpi_q)

#### 9.2 Chuyển dữ liệu ngày sang tháng (Ví dụ 3.5 Dữ liệu tài chính)
Sử dụng `floor_date` từ thư viện `lubridate` để quy về tháng và trích xuất giá trị ngày cuối cùng trong tháng với `slice_tail(n = 1)`.

In [ ]:
library(lubridate)

# Giả định dữ liệu theo ngày
data <- cpi
data$month <- floor_date(data$date, "month")

# Lấy giá cuối tháng
monthly <- data |> 
  group_by(month) |> 
  slice_tail(n = 1)

head(monthly)

#### 9.3 Ghép nhiều chuỗi sau khi chuyển đổi tần suất (Ví dụ 3.6 Ghép GDP và CPI)
Tải chuỗi GDP từ FRED, chuyển sang tần suất quý và ghép cặp cùng với `cpi_q`.

In [ ]:
# Tải dữ liệu GDP và tạo cột quarter để ghép
gdp <- fredr(series_id = "GDP")
gdp$quarter <- as.yearqtr(gdp$date)
gdp <- gdp |> rename(gdp = value)

# Ghép chuỗi GDP và CPI theo cột quarter
data_merge <- merge(gdp, cpi_q, by = "quarter")

# Kiểm tra lại dữ liệu sau khi ghép
dim(data_merge)
head(data_merge)